In [5]:
!pip install -q ultralytics opencv-python-headless

In [6]:
import os

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

INPUT_VIDEO = "yolo_input.mp4"

if not os.path.exists(INPUT_VIDEO):
    if not IN_COLAB:
        raise FileNotFoundError(f"Put {INPUT_VIDEO} next to this notebook and re-run.")
    print("Upload yolo_input.mp4 now...")
    uploaded = files.upload()
    if INPUT_VIDEO not in uploaded:
        vids = [n for n in uploaded if n.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))]
        if not vids:
            raise FileNotFoundError("No video file was uploaded.")
        INPUT_VIDEO = vids[0]

print("Using video:", INPUT_VIDEO)

Using video: yolo_input.mp4


In [7]:
import csv
import cv2
import numpy as np
from ultralytics import YOLO


MODEL_WEIGHTS = "yolov8m.pt"
IMGSZ         = 1280
CONF          = 0.20
MIN_FRAMES    = 8
TRACKER       = "bytetrack.yaml"
OUTPUT_RAW    = "yolo_survivors_raw.mp4"
OUTPUT_FINAL  = "yolo_survivors_detected.mp4"
CSV_PATH      = "survivors.csv"

model = YOLO(MODEL_WEIGHTS)

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open {INPUT_VIDEO}")

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps != fps or fps < 1:
    fps = 30.0
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(OUTPUT_RAW, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
if not out.isOpened():
    raise RuntimeError("Could not open video writer")

font_scale = max(0.5, W / 1600)
thick = max(1, int(W / 800))

seen = {}
confirmed = {}
frame_idx = 0

print(f"[INFO] {INPUT_VIDEO}: {W}x{H} @ {fps:.1f} fps, ~{total} frames. Running {MODEL_WEIGHTS} at imgsz={IMGSZ}")

while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = model.track(
        frame,
        persist=True,
        tracker=TRACKER,
        imgsz=IMGSZ,
        conf=CONF,
        classes=[0],
        verbose=False,
    )

    annotated = frame.copy()
    active = 0
    boxes = results[0].boxes

    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        if boxes.id is not None:
            ids = boxes.id.cpu().numpy().astype(int)
        else:
            ids = np.full(len(xyxy), -1, dtype=int)

        for (x1, y1, x2, y2), c, tid in zip(xyxy, confs, ids):
            tid = int(tid)
            active += 1
            is_confirmed = False
            if tid >= 0:
                seen[tid] = seen.get(tid, 0) + 1
                if seen[tid] >= MIN_FRAMES and tid not in confirmed:
                    confirmed[tid] = (frame_idx / fps, int((x1 + x2) / 2), int((y1 + y2) / 2))
                is_confirmed = tid in confirmed

            color = (0, 0, 255) if is_confirmed else (0, 255, 255)
            tag = "SURVIVOR" if is_confirmed else "candidate"
            label = f"{tag} #{tid} {c:.2f}" if tid >= 0 else f"{tag} {c:.2f}"

            cv2.rectangle(annotated, (int(x1), int(y1)), (int(x2), int(y2)), color, thick + 1)
            cv2.putText(annotated, label, (int(x1), max(int(y1) - 8, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, font_scale * 0.6, color, thick, cv2.LINE_AA)

    cv2.putText(annotated, f"In frame: {active} | Confirmed survivors: {len(confirmed)}",
                (30, int(50 * font_scale) + 10), cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, (255, 255, 255), thick + 2, cv2.LINE_AA)
    cv2.putText(annotated, f"In frame: {active} | Confirmed survivors: {len(confirmed)}",
                (30, int(50 * font_scale) + 10), cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, (0, 0, 255), thick, cv2.LINE_AA)

    out.write(annotated)
    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f"Processed {frame_idx}/{total} frames...")

cap.release()
out.release()

with open(CSV_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["track_id", "first_seen_s", "cx_px", "cy_px"])
    for tid, (t, cx, cy) in sorted(confirmed.items()):
        w.writerow([tid, f"{t:.2f}", cx, cy])

print(f"[DONE] {frame_idx} frames processed. Confirmed survivors: {len(confirmed)}")
for tid, (t, cx, cy) in sorted(confirmed.items()):
    print(f"  survivor #{tid}: first seen at {t:.1f}s, pixel ({cx}, {cy})")

[INFO] yolo_input.mp4: 640x360 @ 25.0 fps, ~341 frames. Running yolov8m.pt at imgsz=1280
Processed 30/341 frames...
Processed 60/341 frames...
Processed 90/341 frames...
Processed 120/341 frames...
Processed 150/341 frames...
Processed 180/341 frames...
Processed 210/341 frames...
Processed 240/341 frames...
Processed 270/341 frames...
Processed 300/341 frames...
Processed 330/341 frames...
[DONE] 341 frames processed. Confirmed survivors: 89
  survivor #1: first seen at 0.3s, pixel (481, 265)
  survivor #2: first seen at 0.3s, pixel (31, 299)
  survivor #3: first seen at 0.3s, pixel (503, 190)
  survivor #4: first seen at 0.3s, pixel (237, 233)
  survivor #5: first seen at 0.3s, pixel (497, 139)
  survivor #6: first seen at 0.3s, pixel (396, 329)
  survivor #7: first seen at 0.3s, pixel (82, 28)
  survivor #8: first seen at 0.3s, pixel (608, 74)
  survivor #9: first seen at 0.3s, pixel (95, 180)
  survivor #10: first seen at 0.3s, pixel (262, 167)
  survivor #11: first seen at 0.3s, p

In [8]:
import shutil, subprocess
from base64 import b64encode
from IPython.display import HTML, display

final_video = OUTPUT_RAW
if shutil.which("ffmpeg"):
    r = subprocess.run(
        ["ffmpeg", "-y", "-i", OUTPUT_RAW,
         "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
         "-c:v", "libx264", "-pix_fmt", "yuv420p", "-loglevel", "error", OUTPUT_FINAL],
        capture_output=True, text=True)
    if r.returncode == 0 and os.path.exists(OUTPUT_FINAL):
        final_video = OUTPUT_FINAL
    else:
        print("ffmpeg re-encode failed, using raw output:", r.stderr[:300])
print("Final video:", final_video, f"({os.path.getsize(final_video)/1e6:.1f} MB)")

if os.path.getsize(final_video) < 25e6:
    data = b64encode(open(final_video, "rb").read()).decode()
    display(HTML(f'<video width="800" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))
else:
    print("Video too large for inline preview - use the download.")

if IN_COLAB:
    files.download(final_video)
    files.download(CSV_PATH)

Final video: yolo_survivors_detected.mp4 (2.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>